In [19]:
import pandas as pd


try:
    df1 = pd.read_csv('/content/dataset-1.csv')
    print("--- Dataset 1 Preview ---")
    print(df1.info())
    display(df1.head())
except Exception as e:
    print(f"Error loading dataset-1.csv: {e}")


try:
    df2 = pd.read_csv('/content/dataset-2.csv')
    print("\n--- Dataset 2 Preview ---")
    print(df2.info())
    display(df2.head())
except Exception as e:
    print(f"Error loading dataset-2.csv: {e}")

--- Dataset 1 Preview ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5634 entries, 0 to 5633
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        5634 non-null   object 
 1   gender            5634 non-null   object 
 2   SeniorCitizen     5634 non-null   int64  
 3   Partner           5634 non-null   object 
 4   Dependents        5634 non-null   object 
 5   tenure            5634 non-null   int64  
 6   PhoneService      5634 non-null   object 
 7   MultipleLines     5634 non-null   object 
 8   InternetService   5634 non-null   object 
 9   OnlineSecurity    5634 non-null   object 
 10  OnlineBackup      5634 non-null   object 
 11  DeviceProtection  5634 non-null   object 
 12  TechSupport       5634 non-null   object 
 13  StreamingTV       5634 non-null   object 
 14  StreamingMovies   5634 non-null   object 
 15  Contract          5634 non-null   object 
 16  PaperlessBilling

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,4950-BDEUX,Male,0,No,No,35,No,No phone service,DSL,No,...,Yes,No,Yes,Yes,Month-to-month,No,Electronic check,49.20,1701.65,False
1,7993-NQLJE,Male,0,Yes,Yes,15,Yes,No,Fiber optic,Yes,...,No,No,No,No,Month-to-month,No,Mailed check,75.10,1151.55,False
2,7321-ZNSLA,Male,0,Yes,Yes,13,No,No phone service,DSL,Yes,...,No,Yes,No,No,Two year,No,Mailed check,40.55,590.35,False
3,4922-CVPDX,Female,0,Yes,No,26,Yes,No,DSL,No,...,Yes,No,Yes,Yes,Two year,Yes,Credit card (automatic),73.50,1905.7,False
4,2903-YYTBW,Male,0,Yes,Yes,1,Yes,No,DSL,No,...,No,No,No,No,Month-to-month,No,Electronic check,44.55,44.55,False



--- Dataset 2 Preview ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3207 entries, 0 to 3206
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   brand         3207 non-null   object
 1   model         3207 non-null   object
 2   model_year    3207 non-null   int64 
 3   milage        3207 non-null   object
 4   fuel_type     3081 non-null   object
 5   engine        3207 non-null   object
 6   transmission  3207 non-null   object
 7   ext_col       3207 non-null   object
 8   int_col       3207 non-null   object
 9   accident      3115 non-null   object
 10  clean_title   2728 non-null   object
 11  price         3207 non-null   object
dtypes: int64(1), object(11)
memory usage: 300.8+ KB
None


,brand,model,model_year,milage,fuel_type,engine,transmission,ext_col,int_col,accident,clean_title,price
0,BMW,M235 i,2015,"81,000 mi.",Gasoline,320.0HP 3.0L Straight 6 Cylinder Engine Gasoli...,6-Speed M/T,White,Red,None reported,Yes,"$27,890"
1,BMW,750 i,2017,"24,650 mi.",Gasoline,445.0HP 4.4L 8 Cylinder Engine Gasoline Fuel,A/T,Silver,Black,None reported,Yes,"$50,000"
2,Aston,Martin V8 Vantage Base,2008,"62,378 mi.",Gasoline,380.0HP 4.3L 8 Cylinder Engine Gasoline Fuel,M/T,Red,Beige,At least 1 accident or damage reported,Yes,"$33,995"
3,INFINITI,QX80 Luxe,2022,"49,860 mi.",Gasoline,5.6L V8 32V GDI DOHC,Automatic,Black Obsidian,Graphite,At least 1 accident or damage reported,Yes,"$47,645"
4,Subaru,BRZ Premium,2023,"5,800 mi.",Gasoline,228.0HP 2.4L 4 Cylinder Engine Gasoline Fuel,6-Speed M/T,Black,Black,None reported,Yes,"$29,990"


In [20]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import xgboost as xgb
import lightgbm as lgb
from sklearn.ensemble import RandomForestClassifier

df = pd.read_csv('/content/dataset-1.csv')


df['TotalCharges'] = pd.to_numeric(df['TotalCharges'].str.strip(), errors='coerce')


df['Churn'] = df['Churn'].astype(int)


X = df.drop(columns=['customerID', 'Churn'])
y = df['Churn']


X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=0.15, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.1765, random_state=42, stratify=y_train_full)

print(f"Train size: {X_train.shape[0]}, Val size: {X_val.shape[0]}, Test size: {X_test.shape[0]}")

Train size: 3942, Val size: 846, Test size: 846


In [21]:
def remove_bad_training_rows(X_tr, y_tr):
    """
    Cleans outliers, duplicates, or anomalous records from the training split only.
    Must return cleaned X_tr and y_tr.
    """
    X_tr = X_tr.copy()
    y_tr = y_tr.copy()


    combined = pd.concat([X_tr, y_tr], axis=1)


    combined = combined.drop_duplicates()


    anomalous = (combined['tenure'] == 0) & (combined['TotalCharges'] > 0)
    combined = combined[~anomalous]

    X_tr_clean = combined.drop(columns=[y_tr.name])
    y_tr_clean = combined[y_tr.name]

    return X_tr_clean, y_tr_clean


X_train_clean, y_train_clean = remove_bad_training_rows(X_train, y_train)
print(f"Cleaned training records count: {X_train_clean.shape[0]} (Original: {X_train.shape[0]})")

Cleaned training records count: 3932 (Original: 3942)


In [22]:

numeric_features = ['tenure', 'MonthlyCharges', 'TotalCharges', 'SeniorCitizen']
categorical_features = [
    'gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
    'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
    'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
    'PaperlessBilling', 'PaymentMethod'
]


numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', drop='first'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

models = {
    'RandomForest': RandomForestClassifier(random_state=42, class_weight='balanced'),
    'XGBoost': xgb.XGBClassifier(random_state=42, eval_metric='logloss'),
    'LightGBM': lgb.LGBMClassifier(random_state=42, verbose=-1, class_weight='balanced')
}

best_f1 = 0
best_model_name = None
best_pipeline = None


for name, model in models.items():
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', model)
    ])


    pipeline.fit(X_train_clean, y_train_clean)

    val_preds = pipeline.predict(X_val)
    val_f1 = f1_score(y_val, val_preds)

    print(f"{name} - Validation Churn F1-Score: {val_f1:.4f}")

    if val_f1 > best_f1:
        best_f1 = val_f1
        best_model_name = name
        best_pipeline = pipeline

print(f"\nBest Model selected: {best_model_name} with F1-score: {best_f1:.4f}")

RandomForest - Validation Churn F1-Score: 0.5280
XGBoost - Validation Churn F1-Score: 0.5553
LightGBM - Validation Churn F1-Score: 0.6362

Best Model selected: LightGBM with F1-score: 0.6362


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [23]:
def evaluate_performance(y_true, y_pred, y_prob, split_name):
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    roc_auc = roc_auc_score(y_true, y_prob)

    print(f"--- {split_name} Split Metrics ---")
    print(f"Accuracy:  {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1-Score:  {f1:.4f}")
    print(f"ROC-AUC:   {roc_auc:.4f}\n")


train_preds = best_pipeline.predict(X_train_clean)
train_probs = best_pipeline.predict_proba(X_train_clean)[:, 1]
evaluate_performance(y_train_clean, train_preds, train_probs, "Training")

val_preds = best_pipeline.predict(X_val)
val_probs = best_pipeline.predict_proba(X_val)[:, 1]
evaluate_performance(y_val, val_preds, val_probs, "Validation")

test_preds = best_pipeline.predict(X_test)
test_probs = best_pipeline.predict_proba(X_test)[:, 1]
evaluate_performance(y_test, test_preds, test_probs, "Test")

/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/ut

--- Training Split Metrics ---
Accuracy:  0.8797
Precision: 0.6939
Recall:    0.9740
F1-Score:  0.8104
ROC-AUC:   0.9615

--- Validation Split Metrics ---
Accuracy:  0.7742
Precision: 0.5567
Recall:    0.7422
F1-Score:  0.6362
ROC-AUC:   0.8503

--- Test Split Metrics ---
Accuracy:  0.7648
Precision: 0.5430
Recall:    0.7054
F1-Score:  0.6136
ROC-AUC:   0.8330



In [24]:
def predict_churn(X_new):
    """
    Accepts custom DataFrame of features and outputs 0/1 predictions and probabilities.
    """
    X_new = X_new.copy()
    if 'customerID' in X_new.columns:
        X_new = X_new.drop(columns=['customerID'])
    if 'TotalCharges' in X_new.columns:
        X_new['TotalCharges'] = pd.to_numeric(X_new['TotalCharges'].astype(str).str.strip(), errors='coerce')

    predictions = best_pipeline.predict(X_new)
    probabilities = best_pipeline.predict_proba(X_new)[:, 1]
    return predictions, probabilities

sample_preds, sample_probs = predict_churn(X_test.head())
print("Sample Predictions:", sample_preds)
print("Sample Probabilities:", sample_probs)

Sample Predictions: [0 0 1 1 1]
Sample Probabilities: [0.09484033 0.431971   0.65081809 0.50445645 0.70865778]


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [25]:
categorical_transformer_dense = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', drop='first', sparse_output=False))
])

preprocessor_dense = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer_dense, categorical_features)
    ]
)

preprocessor_dense.set_output(transform='pandas')

best_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor_dense),
    ('classifier', lgb.LGBMClassifier(random_state=42, verbose=-1, class_weight='balanced'))
])

best_pipeline.fit(X_train_clean, y_train_clean)
print("Re-trained Best LightGBM Pipeline preserving feature names without warnings!")

val_preds = best_pipeline.predict(X_val)
val_probs = best_pipeline.predict_proba(X_val)[:, 1]
evaluate_performance(y_val, val_preds, val_probs, "Validation")

Re-trained Best LightGBM Pipeline preserving feature names without warnings!
--- Validation Split Metrics ---
Accuracy:  0.7742
Precision: 0.5567
Recall:    0.7422
F1-Score:  0.6362
ROC-AUC:   0.8503



### Task 01 - Summary
In this task, we built an end-to-end Machine Learning classification pipeline to predict customer churn for LankaTel. We initiated the process by partitioning our data into train, validation, and test splits to guarantee unbiased evaluation. We handled anomalies and filtered duplicates within the training set using our customized `remove_bad_training_rows()` function. Using scikit-learn's `Pipeline` and `ColumnTransformer`, we integrated robust imputation, scaling, and one-hot encoding seamlessly. After training and validating multiple classifiers, including Random Forest and XGBoost, LightGBM was selected as our champion architecture, demonstrating a strong Churn-class F1-Score of 0.6362 on the validation split and 0.6136 on the test split.